# Introduction to Structured Streaming

**Structured Streaming** is Spark's stream processing engine, built on top of the Spark SQL / DataFrame API. It lets you express continuous, real-time computations using the **same code** you would write for a static batch job.

On Databricks, Structured Streaming is the foundation that powers everything from simple file ingestion (Auto Loader) to production change-data-capture pipelines and Lakeflow Declarative Pipelines.

> The older `DStream` API (Spark Streaming) is **deprecated**. Always use Structured Streaming for new work.

## Why streaming?

Batch jobs answer the question *"what happened up to last midnight?"*. That's enough for many reports, but not for use cases where freshness matters:

- **Operational dashboards** — sales, latency, error rates updated within seconds.
- **Fraud and anomaly detection** — react before damage is done, not the next morning.
- **Continuous ingestion** — process files as they land in cloud storage, not in a nightly sweep.
- **Change Data Capture (CDC)** — replicate database changes into the lakehouse with low latency.
- **IoT and clickstream** — high-volume event streams where buffering 24h is impractical.

The trade-off is that streams introduce *new* problems: out-of-order events, partial failures mid-batch, ever-growing state, and the need to recover from exactly where you stopped.

## The core idea — the unbounded table

Structured Streaming's key insight is to model a stream as a **table that never stops growing**.

```
        ┌──────────────────────── unbounded input table ──────────────────────────┐
        │  row 1   row 2   row 3   row 4   row 5   row 6   ...   (new rows ──▶)  │
        └────────────────────────────────────────────────────────────────────────┘
                                            │
                                            ▼
                              ┌───────────────────────┐
                              │  same DataFrame API   │
                              │  select / filter /    │
                              │  groupBy / join / ... │
                              └───────────────────────┘
                                            │
                                            ▼
        ┌──────────────────────── result table ────────────────────────────────────┐
        │           continuously updated as new input rows arrive                  │
        └─────────────────────────────────────────────────────────────────────────┘
```

Every time new data arrives, Spark conceptually re-runs your query against the (now larger) input table and updates the result. In practice it does this **incrementally** — only processing the new rows — but the mental model is what makes the API feel familiar.

**Implication:** if you can write the batch query, you can usually write the streaming query. The same `df.groupBy(...).count()` works on both.

## Batch vs Streaming — the API difference

The transformations are identical. Only how you **read** and **write** changes:

| Step | Batch | Streaming |
|---|---|---|
| Read | `spark.read.format(...).load(...)` | `spark.readStream.format(...).load(...)` |
| Transform | `df.filter(...).groupBy(...).agg(...)` | *(exactly the same)* |
| Write | `df.write.format(...).save(...)` | `df.writeStream.format(...).start(...)` |
| Returns | nothing — runs and finishes | a `StreamingQuery` handle — keeps running |

That `.start()` is where everything changes: the query becomes a long-running process that you can monitor, pause, and stop.

## Execution modes — micro-batch vs continuous

Structured Streaming offers two execution modes:

| Mode | How it works | Latency | Status |
|---|---|---|---|
| **Micro-batch** *(default)* | Spark wakes up on a trigger, reads any new input, runs a small batch job, writes the output, commits the checkpoint, then sleeps until the next trigger. | ~100 ms – seconds | Stable, recommended |
| **Continuous** | A long-running task per partition processes records one at a time. | ~1 ms | Experimental, limited operator support |

**Use micro-batch unless you have a hard sub-second latency requirement.** It supports the full DataFrame API, exactly-once semantics, and all the sinks. Continuous mode is rarely needed and supports only a narrow subset of operations.

## Anatomy of a streaming query

Every Structured Streaming pipeline has the same five pieces:

```
  ┌────────────┐    ┌────────────────┐    ┌────────────┐    ┌────────────┐
  │   Source   │──▶ │ Transformations│──▶ │    Sink    │    │ Checkpoint │
  │ (input)    │    │ (DataFrame ops)│    │  (output)  │    │ (recovery) │
  └────────────┘    └────────────────┘    └────────────┘    └────────────┘
        ▲                                        ▲                ▲
        └──── Trigger ──── decides when ─────────┴────────────────┘
             (processingTime / availableNow / once / continuous)
```

1. **Source** — where rows come from (files in a Volume, Kafka topic, Delta table, rate source).
2. **Transformations** — the same DataFrame logic you would write for batch.
3. **Sink** — where results go (Delta table, files, foreachBatch, console, memory).
4. **Trigger** — controls *when* a micro-batch fires.
5. **Checkpoint** — a directory where Spark records progress and state, so the query can resume after a restart.

## Key concepts you will meet again

| Concept | Meaning | Notebook |
|---|---|---|
| **Source** | A streaming input — file source, Auto Loader (`cloudFiles`), Delta, Kafka, rate. | 02 |
| **Sink** | A streaming output — Delta table, files, `foreachBatch`, console, memory. | 06, 08 |
| **Trigger** | Cadence of micro-batches: `processingTime='10 seconds'`, `availableNow=True`, `once=True`, continuous. | 06 |
| **Output mode** | What gets written each batch: **append**, **update**, or **complete**. | 06 |
| **Checkpoint location** | Required directory where progress and state are stored — the only thing that makes restarts safe. | 06 |
| **Watermark** | A bound on how late event-time data can be before it is dropped — required for stateful aggregations and stream-stream joins. | 04 |
| **State store** | Where stateful operators (windows, dedup, stream-stream joins) keep their working data between batches. | 04, 05 |
| **Exactly-once** | A guarantee that each input record affects the sink at most once, achieved via checkpoints + idempotent sinks (Delta is one). | 07, 08 |

## Output modes at a glance

When you call `.writeStream.outputMode(...)`, you're telling Spark *what* to emit each micro-batch:

| Mode | Emits | Typical use |
|---|---|---|
| **append** *(default)* | Only **new** rows added to the result since the last batch. | Filter / map / projection pipelines, append-only ingestion. |
| **update** | Only rows whose value **changed** in this batch. | Aggregations where you want incremental updates (e.g. running counts). |
| **complete** | The **entire** result table, every batch. | Small aggregations where you want the full state each time (e.g. dashboard totals). |

Not every mode works with every query. For example, a `groupBy(...).count()` without a watermark cannot use `append` mode, because Spark cannot know when a group is "final". The compiler will tell you if you pick an incompatible mode.

## Checkpointing — non-negotiable in production

A checkpoint location is a directory where Structured Streaming records:

- **Offsets** — which input rows have been read and committed.
- **State** — values held by stateful operators (windows, deduplication, joins).
- **Commit log** — which batches have been written to the sink.

When a query restarts (intentional or after a crash), it reads the checkpoint and resumes exactly where it stopped. Without a checkpoint, every restart re-reads everything from scratch — and stateful queries cannot survive a restart at all.

```python
(df.writeStream
   .format("delta")
   .option("checkpointLocation", "/Volumes/main/streaming/_checkpoints/orders")
   .toTable("main.streaming.orders_clean"))
```

**Rules of thumb:**

- One checkpoint directory **per streaming query**. Never share between queries.
- Put it on durable cloud storage (a Unity Catalog Volume, ADLS, S3) — never on local disk or `/tmp`.
- Treat the checkpoint as part of the query's identity. Deleting it means starting over.

## What makes streaming genuinely hard

Most surprises in streaming come from a small set of recurring problems:

| Problem | What it looks like | Where we address it |
|---|---|---|
| **Late and out-of-order events** | An event with a timestamp from 10 minutes ago arrives now — should it still update an old window? | Notebook 04 (watermarks) |
| **Unbounded state growth** | A `groupBy` over a never-ending stream eventually keeps state for every key ever seen, exhausting memory. | Notebook 04 |
| **Exactly-once writes** | A crash mid-batch must not produce duplicate output rows after recovery. | Notebook 06 (checkpoints), 08 (`foreachBatch` + MERGE) |
| **Schema evolution** | The producer adds a new column. Does the stream fail, drop it, or pick it up? | Notebook 02 (Auto Loader), 09 |
| **Backfill and restart** | Re-run history without disturbing the live stream; resume cleanly after long downtime. | Notebook 06, 09 |
| **Monitoring** | A stream that quietly stops processing is worse than one that fails loudly. | Notebook 09 |

## What Databricks adds on top

Open-source Structured Streaming is the engine. Databricks layers several features that make it dramatically easier to use in production:

| Feature | What it gives you |
|---|---|
| **Auto Loader (`cloudFiles`)** | Incremental, scalable file ingestion with built-in schema inference and evolution — covered in notebook 02. |
| **Delta Lake as source and sink** | ACID streaming reads and writes, schema enforcement, and Change Data Feed (CDF) — covered in notebook 07. |
| **`Trigger.AvailableNow`** | Process all currently-available data, then stop — perfect for cost-efficient "streaming on a schedule". |
| **Lakeflow Declarative Pipelines** | A higher-level declarative wrapper over Structured Streaming for managed ELT pipelines (formerly Delta Live Tables). |
| **Unity Catalog Volumes** | Governed, durable storage for both source data and checkpoints. |

## What's coming in the next notebooks

This series walks the same path a real streaming pipeline takes — from picking a source to running it reliably in production:

| Notebook | Topic |
|---|---|
| **02** | Streaming sources and Auto Loader — file source, rate source, `cloudFiles` schema inference and evolution |
| **03** | Streaming transformations — what works on a stream and what doesn't |
| **04** | Windowing and watermarks — event-time aggregations and bounding state |
| **05** | Streaming joins — stream-static and stream-stream joins |
| **06** | Output modes, triggers and checkpoints — controlling cadence and recovery |
| **07** | Streaming with Delta Lake — Delta as source and sink, Change Data Feed |
| **08** | `foreachBatch` and custom sinks — MERGE, multi-sink fan-out, idempotent writes |
| **09** | Monitoring and production patterns — listeners, metrics, recovery, common pitfalls |